# Create Extracellular Recording Array

## Local Setup

This notebook requires [BlueRecording](https://github.com/openbraininstitute/BlueRecording).
You'll need the `katta/circuit-config-support` branch.

### 1. Install BlueRecording

```bash
git clone https://github.com/openbraininstitute/BlueRecording.git
cd BlueRecording
git checkout katta/circuit-config-support
./dev_setup.sh
source env.sh
```

This activates an environment with BlueRecording and all its dependencies.

### 2. Install obi-one

With the BlueRecording environment still active, navigate to your `obi-one` folder and run:

```bash
pip install -e ".[notebooks]"
```

### 3. Run the notebook

```bash
jupyter lab examples/K_create_recording_array/create_recording_array.ipynb
```

In [ ]:
import obi_one as obi
from entitysdk import Client, ProjectContext

from obi_auth import get_token

virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(api_url="https://staging.openbraininstitute.org/api/entitycore", token_manager=token, project_context=project_context)

In [ ]:
circuit_id = "7d007c43-201f-42d6-960d-93f6229fe935"

In [ ]:
scan_config = obi.CreateExtracellularRecordingArrayScanConfig.empty_config()

info = obi.Info(campaign_name="Campaign Name", campaign_description="Campaign Description")
scan_config.set(info, name="info")

scan_config_initialize = obi.CreateExtracellularRecordingArrayScanConfig.Initialize(circuit=obi.CircuitFromID(id_str=circuit_id), 
                                                            # calculation_method=["PointSource", "LineSource", "ObjectiveCSD"])
                                                            calculation_method="LineSource")
scan_config.set(scan_config_initialize, name='initialize')

# Electrode positions. `electrode_locations` is a dictionary of blocks, so multiple probes
# can be added (each contributes its electrodes to the array). LinearExtracellularLocations
# places `n_electrodes` along a line spaced by `spacing` micrometres; add an
# obi.Neuropixels1ExtracellularLocations(...) block for a Neuropixels 1.0 layout.
electrode_locations = obi.LinearExtracellularLocations(
    n_electrodes=16,
    spacing=20.0,
    origin_x=0.0,
    origin_y=0.0,
    origin_z=0.0,
)
scan_config.add(electrode_locations, name="LinearProbe")

validated_sim_conf = scan_config.validated_config()

In [ ]:
grid_scan = obi.GridScanGenerationTask(form=validated_sim_conf, coordinate_directory_option="ZERO_INDEX", output_root='../../../../obi-output/create_recording_array/grid_scan')
grid_scan.multiple_value_parameters(display=True)
grid_scan.coordinate_parameters(display=True)
grid_scan.execute(db_client=db_client)
obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)

# Download the electrode array weight matrix

Fetches the recording-array result (`TaskResult` of type
`extracellular_recording_weights_calculation__result`) from staging by ID and downloads its
`electrode_array_weight_matrix` asset (the HDF5 weights file).

In [ ]:
from pathlib import Path
from uuid import UUID

from entitysdk.models import SimulatableExtracellularRecordingArray
from entitysdk.types import AssetLabel

# TaskResult of type extracellular_recording_weights_calculation__result on staging.
recording_array_id = UUID("97d1d91f-3437-4b6f-8494-08b637f55235")
recording_array = db_client.get_entity(
    entity_id=recording_array_id,
    entity_type=SimulatableExtracellularRecordingArray,
)

# Select and download the electrode_array_weight_matrix asset.
weights_asset = db_client.select_assets(
    entity=recording_array,
    selection={"label": AssetLabel.electrode_array_weight_matrix},
).one()

output_dir = Path("../../../obi-output/extracellular_recording_weights")
output_dir.mkdir(parents=True, exist_ok=True)
weights_path = db_client.download_file(
    entity_id=recording_array.id,
    entity_type=SimulatableExtracellularRecordingArray,
    asset_id=weights_asset.id,
    output_path=output_dir,
)
print(f"Downloaded electrode_array_weight_matrix to: {weights_path}")